# Hypothesis 2: tree-path features add value beyond random projection

We compare direct random projection, real one-shot tree paths, full Forest Sketch, and a random sparse control matched to the path matrix shape. The hypothesis is supported only when real paths beat both direct projection and the matched random control.

In [1]:
import sys
from pathlib import Path

notebooks_dir = Path.cwd() / 'notebooks'
if not (notebooks_dir / '_hypothesis_utils.py').exists():
    notebooks_dir = Path.cwd()
sys.path.insert(0, str(notebooks_dir))

import pandas as pd
from _hypothesis_utils import (
    CLASSIFICATION_SEEDS, MatchedRandomSparsePathTransformer,
    OneShotTreePathTransformer, classification_data, classification_score,
    downstream_classifier, forest_classifier, forest_sketch, initial_projection,
)

In [2]:
rows = []
for seed in CLASSIFICATION_SEEDS[:2]:
    X_train, _, X_test, y_train, _, y_test = classification_data(seed)
    X_train_0, X_test_0, _ = initial_projection(X_train, X_test, 32, seed)
    methods = [
        ('initial random projection', X_train_0, X_test_0),
        ('one-shot tree-path', OneShotTreePathTransformer(forest_classifier(seed), 32, seed), None),
        ('random sparse control', MatchedRandomSparsePathTransformer(forest_classifier(seed), 32, seed), None),
        ('Forest Sketch', forest_sketch(seed, 32, 2), None),
    ]
    for name, train_view, test_view in methods:
        if test_view is None:
            train_view.fit(X_train, y_train)
            test_view = train_view.transform(X_test)
            train_view = train_view.transform(X_train)
        accuracy, errors = classification_score(
            downstream_classifier(seed), train_view, y_train, test_view, y_test
        )
        rows.append({'seed': seed, 'method': name, 'accuracy': accuracy, 'errors': errors})
results = pd.DataFrame(rows)
display(results.round(3))
summary = results.groupby('method').accuracy.agg(['mean', 'std']).round(3)
display(summary)
path_mean = summary.loc['one-shot tree-path', 'mean']
supported = path_mean > summary.loc['initial random projection', 'mean'] and path_mean > summary.loc['random sparse control', 'mean']
print(f'Hypothesis 2: {"SUPPORTED" if supported else "NOT SUPPORTED"} — real path mean accuracy={path_mean:.3f}')

,seed,method,accuracy,errors
0,0,initial random projection,0.611,171
1,0,one-shot tree-path,0.661,149
2,0,random sparse control,0.523,210
3,0,Forest Sketch,0.605,174
4,1,initial random projection,0.684,139
5,1,one-shot tree-path,0.782,96
6,1,random sparse control,0.491,224
7,1,Forest Sketch,0.666,147


,mean,std
method,,
Forest Sketch,0.635,0.043
initial random projection,0.648,0.051
one-shot tree-path,0.722,0.085
random sparse control,0.507,0.022


Hypothesis 2: SUPPORTED — real path mean accuracy=0.722


## Conclusion

**Hypothesis:** projected real tree-path features improve performance over direct random projection and matched random sparse features.

**Experiment:** compare initial random projection, one-shot real tree paths, full Forest Sketch, and a random sparse control with the same output dimension across two seeds.

**Measure:** held-out classification accuracy and integer error count from the same downstream logistic-regression model.

**Result:** supported in this run: the real path representation achieved mean accuracy 0.722 and exceeded both comparison controls.